# 07 - Modern CNN Architectural Innovations

This notebook covers state-of-the-art modern CNN building blocks developed to optimize parameter efficiency, spatial attention, and capacity (excluding pre-trained transfer learning):

---

## Modern Innovations Covered:
1. **Depthwise Separable Convolutions (MobileNet Paradigm)**: Splitting standard 2D conv into Spatial Depthwise Conv + Pointwise 1x1 Conv.
2. **Squeeze-and-Excitation (SE) Channel Attention**: Adaptive channel re-weighting using Global Average Pooling.
3. **Grouped Convolutions (`groups` parameter in `nn.Conv2d`)**: ResNeXt parallel feature channel groups.
4. **ConvNeXt Block (A 2020s Modern Pure CNN)**: $7 \times 7$ Depthwise Conv, LayerNorm in 2D, GELU, and Inverted Bottleneck ratio.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f"PyTorch Version: {torch.__version__}")


## 1. Depthwise Separable Convolutions (MobileNet)

A standard $3 \times 3$ convolution performs spatial and cross-channel filtering simultaneously, taking $C_{in} \times C_{out} \times K \times K$ parameters.

Depthwise Separable Convolution decouples this into two steps:
1. **Depthwise Conv**: $3 \times 3$ spatial convolution per channel independently (`groups=in_channels`).
2. **Pointwise Conv**: $1 \times 1$ convolution combining channels.

Parameter reduction factor:
$$\frac{\text{Depthwise Separable}}{\text{Standard}} \approx \frac{1}{C_{out}} + \frac{1}{K^2}$$


In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # 1. Depthwise Conv (groups = in_channels)
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=3, stride=stride, padding=1, groups=in_channels, bias=False
        )
        # 2. Pointwise Conv (1x1)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.relu(self.bn(x))
        return x

# Parameter comparison: Standard vs Depthwise Separable
in_c, out_c = 128, 256
std_conv = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False)
dw_sep_conv = DepthwiseSeparableConv(in_c, out_c)

std_params = std_conv.weight.numel()
dw_params = dw_sep_conv.depthwise.weight.numel() + dw_sep_conv.pointwise.weight.numel()

print(f"Standard Conv2d (128 -> 256, 3x3) Parameters:           {std_params:,}")
print(f"Depthwise Separable Conv2d (128 -> 256, 3x3) Parameters: {dw_params:,}")
print(f"Parameter Reduction: {((1 - dw_params / std_params) * 100):.2f}% fewer parameters!")


## 2. Squeeze-and-Excitation (SE) Channel Attention Blocks (SENet)

Squeeze-and-Excitation dynamically recalibrates channel-wise feature responses:
1. **Squeeze**: Global Average Pooling collapses $H \times W$ into a $1 \times 1 \times C$ vector.
2. **Excitation**: Bottleneck MLP (`Linear -> ReLU -> Linear -> Sigmoid`) generates channel attention weights $s \in [0, 1]^C$.
3. **Scale**: Multiply original feature maps by channel weights: $\tilde{x}_c = s_c \cdot x_c$.


In [ ]:
class SqueezeAndExcitationBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super().__init__()
        self.global_pool = nn.AdaptiveAvgPool2d(1) # Squeeze: HxW -> 1x1
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction_ratio, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction_ratio, channels, bias=False),
            nn.Sigmoid() # Excitation: [0, 1] channel weights
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        # Squeeze
        squeeze = self.global_pool(x).view(b, c)
        # Excitation
        excitation = self.fc(squeeze).view(b, c, 1, 1)
        # Scale channels
        return x * excitation

se = SqueezeAndExcitationBlock(channels=64, reduction_ratio=16)
x_in = torch.randn(2, 64, 32, 32)
x_out = se(x_in)

print(f"SE Block Input Shape:  {list(x_in.shape)}")
print(f"SE Block Output Shape: {list(x_out.shape)}")


## 3. Grouped Convolutions (ResNeXt)

`groups=G` divides $C_{in}$ and $C_{out}$ into $G$ independent groups. Each group computes convolutions over its slice of channels.


In [ ]:
# Grouped Conv with 4 groups (ResNeXt concept)
grouped_conv = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1, groups=4)
x_grp = torch.randn(2, 64, 16, 16)

print(f"Grouped Conv Input:  {list(x_grp.shape)}")
print(f"Grouped Conv Output: {list(grouped_conv(x_grp).shape)}")
print(f"Parameters in Grouped Conv (groups=4): {grouped_conv.weight.numel():,}")
print(f"Parameters in Standard Conv (groups=1): {nn.Conv2d(64, 64, 3, padding=1).weight.numel():,}")


## 4. ConvNeXt Block: Modernizing Pure CNNs for the 2020s

ConvNeXt (Liu et al., 2022) modernizes standard CNNs to match Vision Transformer (ViT) performance:
1. **$7 \times 7$ Depthwise Convolution** (larger spatial receptive field per block).
2. **Channel Permuted 2D LayerNorm** (replaces BatchNorm).
3. **$1 \times 1$ Inverted Bottleneck** ($C \rightarrow 4C \rightarrow C$).
4. **GELU Activation** (replaces ReLU).


In [ ]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 1. 7x7 Depthwise Conv
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        # 2. LayerNorm (performed over channel dimension)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        # 3. 1x1 Conv Inverted Bottleneck (dim -> 4*dim -> dim)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        
        # Permute (B, C, H, W) -> (B, H, W, C) for LayerNorm & Linear layers
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        # Permute back to (B, C, H, W)
        x = x.permute(0, 3, 1, 2)
        
        return residual + x

convnext_block = ConvNeXtBlock(dim=96)
x_cnext = torch.randn(2, 96, 28, 28)
out_cnext = convnext_block(x_cnext)

print(f"ConvNeXt Block Input Shape:  {list(x_cnext.shape)}")
print(f"ConvNeXt Block Output Shape: {list(out_cnext.shape)}")
print("ConvNeXt integrates 7x7 depthwise conv, 2D LayerNorm, GELU, and inverted bottleneck!")
